In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import joblib

In [2]:
# === Load dataset ===
df = pd.read_csv("ISP-bottleneck-dataset.csv")

# === Drop timestamp ===
df.drop(columns=["timestamp"], inplace=True)

# === Normalize label column (lowercase all labels) ===
df["label"] = df["label"].str.strip().str.lower()

df.info()
df['label'].value_counts()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 636 entries, 0 to 635
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   packet_loss_ratio       636 non-null    float64
 1   latency_jitter_ratio    636 non-null    float64
 2   dns_resolve_time_ratio  636 non-null    float64
 3   hop_count_ratio         636 non-null    float64
 4   per_hop_rtt_ratio       636 non-null    float64
 5   label                   636 non-null    object 
dtypes: float64(5), object(1)
memory usage: 29.9+ KB


,packet_loss_ratio,latency_jitter_ratio,dns_resolve_time_ratio,hop_count_ratio,per_hop_rtt_ratio
count,636.000000,636.000000,636.000000,636.000000,636.000000
mean,2.904868,1.413133,2.161720,1.140359,1.358179
std,8.246040,1.793518,4.917008,0.624348,1.235796
min,0.000000,0.000000,0.001639,0.000000,0.000000
25%,0.000000,0.392025,0.036749,1.000000,0.553398
50%,0.000000,0.931594,0.346977,1.000000,0.839120
75%,1.392074,1.558795,1.584809,1.041667,1.845796
max,49.889185,9.957731,44.236584,4.898120,7.452202


In [3]:
# === Encode target labels ===
label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["label"])
joblib.dump(label_encoder, 'label_encoderISP.pkl')
X = df.drop(columns=["label", "label_encoded"])
y = df["label_encoded"]

In [4]:
# === Train-test split (Stratified) ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(xgb.__version__)

3.0.2


In [5]:
model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=6,
    eval_metric="mlogloss",
    use_label_encoder=False,
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    #subsample=0.8,
    #colsample_bytree=0.8,
    random_state=42,
    tree_method='exact'
)

# Train without early stopping
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=True)

[0]	validation_0-mlogloss:1.39701
[1]	validation_0-mlogloss:1.23299
[2]	validation_0-mlogloss:1.09881
[3]	validation_0-mlogloss:0.98464
[4]	validation_0-mlogloss:0.88748
[5]	validation_0-mlogloss:0.80381
[6]	validation_0-mlogloss:0.73297
[7]	validation_0-mlogloss:0.67120
[8]	validation_0-mlogloss:0.61647
[9]	validation_0-mlogloss:0.56932
[10]	validation_0-mlogloss:0.52797
[11]	validation_0-mlogloss:0.49012
[12]	validation_0-mlogloss:0.45865
[13]	validation_0-mlogloss:0.43041
[14]	validation_0-mlogloss:0.40540
[15]	validation_0-mlogloss:0.38281
[16]	validation_0-mlogloss:0.36222
[17]	validation_0-mlogloss:0.34397
[18]	validation_0-mlogloss:0.32634
[19]	validation_0-mlogloss:0.31022
[20]	validation_0-mlogloss:0.29615
[21]	validation_0-mlogloss:0.28358
[22]	validation_0-mlogloss:0.27145
[23]	validation_0-mlogloss:0.26055
[24]	validation_0-mlogloss:0.25117
[25]	validation_0-mlogloss:0.24311
[26]	validation_0-mlogloss:0.23586
[27]	validation_0-mlogloss:0.22993
[28]	validation_0-mlogloss:0.2

c:\Users\msaad\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:14:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[74]	validation_0-mlogloss:0.13346
[75]	validation_0-mlogloss:0.13408
[76]	validation_0-mlogloss:0.13409
[77]	validation_0-mlogloss:0.13458
[78]	validation_0-mlogloss:0.13446
[79]	validation_0-mlogloss:0.13503
[80]	validation_0-mlogloss:0.13509
[81]	validation_0-mlogloss:0.13501
[82]	validation_0-mlogloss:0.13476
[83]	validation_0-mlogloss:0.13492
[84]	validation_0-mlogloss:0.13511
[85]	validation_0-mlogloss:0.13502
[86]	validation_0-mlogloss:0.13497
[87]	validation_0-mlogloss:0.13505
[88]	validation_0-mlogloss:0.13514
[89]	validation_0-mlogloss:0.13496
[90]	validation_0-mlogloss:0.13502
[91]	validation_0-mlogloss:0.13486
[92]	validation_0-mlogloss:0.13469
[93]	validation_0-mlogloss:0.13477
[94]	validation_0-mlogloss:0.13494
[95]	validation_0-mlogloss:0.13416
[96]	validation_0-mlogloss:0.13344
[97]	validation_0-mlogloss:0.13268
[98]	validation_0-mlogloss:0.13198
[99]	validation_0-mlogloss:0.13175


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [6]:
# === Predict and evaluate ===
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)  # Confidence scores

# === Display results ===
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

         dns       0.92      0.92      0.92        13
     latency       0.92      0.86      0.89        14
        loss       1.00      0.94      0.97        17
      normal       0.96      0.99      0.97        73
  traceroute       1.00      1.00      1.00        11

    accuracy                           0.96       128
   macro avg       0.96      0.94      0.95       128
weighted avg       0.96      0.96      0.96       128

Confusion Matrix:
[[12  0  0  1  0]
 [ 0 12  0  2  0]
 [ 0  1 16  0  0]
 [ 1  0  0 72  0]
 [ 0  0  0  0 11]]


In [7]:
# === Example: Print prediction with confidence ===
for i in range(5):
    probs = y_proba[i]
    pred_class = np.argmax(probs)
    conf = probs[pred_class]
    print(f"Prediction: {label_encoder.inverse_transform([pred_class])[0]}, Confidence: {conf:.2f}")
    
model.save_model("bn_model_1ISPXGB.json")  # or .bin

Prediction: normal, Confidence: 1.00
Prediction: normal, Confidence: 0.92
Prediction: normal, Confidence: 1.00
Prediction: loss, Confidence: 0.93
Prediction: normal, Confidence: 1.00
